In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[18]:


#!/usr/bin/env python
# coding: utf-8

"""
Оценка качества лучшего MCE-регрессора отдельно для каждого магнитного поля.

Входы задаются в разделе НАСТРОЙКИ:
1) исходный DB_mc.xlsx;
2) сохранённая лучшая модель best_overall_model.joblib.

Скрипт воспроизводит подготовку Stoichiometry/XenonPy/Combined-признаков,
использованную при обучении. Для оценки качества сначала используется таблица
outer-OOF прогнозов, сохранённая основным расчётом. Если она отсутствует,
выполняется 5-fold group CV зафиксированной конфигурации модели. Записи одного
состава не могут одновременно попадать в train и test.
"""

# Ограничиваем число потоков до импорта NumPy.
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymatgen.core import Composition
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Эти импорты нужны также для загрузки joblib-моделей соответствующих классов.
try:
    import xgboost  # noqa: F401
except ImportError:
    xgboost = None

try:
    import catboost  # noqa: F401
except ImportError:
    catboost = None

from xenonpy.descriptor.compositions import (
    MaxPooling,
    MinPooling,
    WeightedAverage,
    WeightedSum,
    WeightedVariance,
)


# =============================================================================
# НАСТРОЙКИ: ИЗМЕНИТЕ ТОЛЬКО ПУТИ ПЕРЕД ЗАПУСКОМ
# =============================================================================

INPUT_FILE = Path(
    r"/Users/artemykagramanian/Desktop/ампермагнит/MCE + ML/итоговые датасеты/DB_mc_merged.xlsx"
)

INPUT_SHEET = "1"

OUTPUT_DIRECTORY = Path(
    r"/Users/artemykagramanian/Desktop/ампермагнит/MCE + ML/ipynb and results FMM version/MCE_Three_Feature_Sets_GridSearch_results\models\best_overall_model.joblib"
)

# None: папка fieldwise_quality создаётся рядом с папками models и tables.
OUTPUT_DIRECTORY = None

# Поле включается в анализ, только если N строго больше этого значения.
MIN_SAMPLES_PER_FIELD = 10
FIELD_ROUND_DECIMALS = 3

# Поле, для которого строятся расширенные диагностические графики.
DIAGNOSTIC_FIELD_T = 2.0
TOP_ERROR_LABELS = 8

# Размер кристаллитов является непрерывным признаком, поэтому записи делятся
# на квантильные интервалы с близким числом наблюдений.
CRYSTALLITE_SIZE_BINS = 5
MIN_SAMPLES_PER_SIZE_BIN = 10

# Число групповых фолдов для запасного расчёта, если outer-OOF CSV не найден.
CV_SPLITS = 5
RANDOM_STATE = 42
XENONPY_BATCH_SIZE = 20

TARGET_PROPERTY = "ME (J/kg.K)"
TARGET_COLUMN = "target_entropy_change"
TARGET_AS_ABSOLUTE = True
OXYGEN_REFERENCE = 3.0
STOICHIOMETRY_DECIMALS = 12

FIGURE_SIZE_CM = 8.0
FIGURE_SIZE_INCH = FIGURE_SIZE_CM / 2.54

COLORS = {
    "points": "#0072B2",
    "line": "#D55E00",
    "reference": "#222222",
    "grid": "#B8B8B8",
}

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 8,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "figure.dpi": 130,
})


# =============================================================================
# ПОДГОТОВКА ИСХОДНОЙ ВЫБОРКИ
# =============================================================================

def numeric_from_mixed(value):
    """Извлекает число из 5, '5 T', '45 nm' или '5,0 Тл'."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip().replace("−", "-").replace(",", ".")
    match = re.search(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", text
    )
    return float(match.group(0)) if match else np.nan


def normalize_property_name(value):
    return re.sub(r"\s+", "", str(value).strip()).casefold()


def clean_formula(formula):
    if pd.isna(formula):
        return None
    result = (
        str(formula).strip().replace(" ", "").replace("−", "-")
        .replace("–", "-").replace(",", ".")
    )
    return result or None


def parse_formula(formula):
    cleaned = clean_formula(formula)
    if cleaned is None:
        raise ValueError("пустая химическая формула")
    try:
        parsed = Composition(cleaned).get_el_amt_dict()
    except Exception as error:
        raise ValueError(f"ошибка разбора формулы: {error}") from error
    result = {str(element): float(amount) for element, amount in parsed.items()}
    if not result or any(not np.isfinite(v) or v <= 0 for v in result.values()):
        raise ValueError("состав содержит некорректные коэффициенты")
    return result


def normalize_composition_to_o3(composition):
    oxygen = float(composition.get("O", np.nan))
    if not np.isfinite(oxygen) or oxygen <= 0:
        raise ValueError("в составе отсутствует положительный коэффициент O")
    scale = OXYGEN_REFERENCE / oxygen
    normalized = {}
    for element, amount in composition.items():
        value = float(amount) * scale
        nearest_integer = round(value)
        if np.isclose(
            value,
            nearest_integer,
            atol=10 ** (-STOICHIOMETRY_DECIMALS),
            rtol=0.0,
        ):
            value = float(nearest_integer)
        else:
            value = float(round(value, STOICHIOMETRY_DECIMALS))
        if value > 0:
            normalized[str(element)] = value
    normalized["O"] = OXYGEN_REFERENCE
    return normalized


def format_amount(value, omit_one=True):
    value = float(value)
    if omit_one and np.isclose(value, 1.0, atol=1e-12, rtol=0.0):
        return ""
    if np.isclose(value, round(value), atol=1e-12, rtol=0.0):
        return str(int(round(value)))
    return np.format_float_positional(
        value, precision=STOICHIOMETRY_DECIMALS, unique=True, trim="-"
    )


def composition_to_formula(composition):
    elements = sorted(element for element in composition if element != "O")
    if "O" in composition:
        elements.append("O")
    return "".join(
        f"{element}{format_amount(composition[element], element != 'O')}"
        for element in elements
    )


def composition_key(composition):
    return "|".join(
        f"{element}:{float(composition[element]):.12g}"
        for element in sorted(composition)
    )


def load_raw_source():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Исходный файл не найден: {INPUT_FILE}")
    if INPUT_FILE.suffix.lower() in {".xlsx", ".xls"}:
        source = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)
    elif INPUT_FILE.suffix.lower() in {".csv", ".txt"}:
        source = pd.read_csv(INPUT_FILE)
    else:
        raise ValueError("Поддерживаются файлы XLSX, XLS и CSV.")
    source.columns = source.columns.astype(str).str.strip()
    required = {
        "Compound", "Property", "Value", "Magnetic Field", "D_crystalite [nm]"
    }
    missing = required.difference(source.columns)
    if missing:
        raise KeyError(f"В исходном файле отсутствуют столбцы: {sorted(missing)}")
    source = source.copy()
    source["Source_row"] = np.arange(len(source)) + 2
    return source


def prepare_ml_dataset(source):
    """Повторяет фильтрацию и агрегацию обучающего скрипта."""
    property_mask = source["Property"].map(normalize_property_name).eq(
        normalize_property_name(TARGET_PROPERTY)
    )
    selected = source.loc[property_mask].copy()
    selected["target"] = selected["Value"].map(numeric_from_mixed)
    selected["field"] = selected["Magnetic Field"].map(numeric_from_mixed)
    selected["size"] = selected["D_crystalite [nm]"].map(numeric_from_mixed)
    if TARGET_AS_ABSOLUTE:
        selected["target"] = selected["target"].abs()

    valid_rows = []
    rejected_rows = []
    for _, row in selected.iterrows():
        try:
            composition = parse_formula(row["Compound"])
            if float(composition.get("Mn", 0.0)) <= 0:
                raise ValueError("соединение не содержит Mn")
            normalized = normalize_composition_to_o3(composition)
            target = float(row["target"])
            field = float(row["field"])
            size = float(row["size"])
            if not np.isfinite(target) or target <= 0:
                raise ValueError("целевая величина отсутствует или неположительна")
            if not np.isfinite(field) or field <= 0:
                raise ValueError("магнитное поле отсутствует или неположительно")
            if not np.isfinite(size) or size <= 0:
                raise ValueError("размер кристаллитов отсутствует или неположителен")
        except (TypeError, ValueError) as error:
            rejected_rows.append({
                "Source_row": row["Source_row"],
                "Compound": row["Compound"],
                "Rejection_reason": str(error),
            })
            continue

        valid_rows.append({
            "Source_row": int(row["Source_row"]),
            "original_formula": str(row["Compound"]).strip(),
            "normalized_formula_O3": composition_to_formula(normalized),
            "composition_key": composition_key(normalized),
            "composition": normalized,
            "magnetic_field_T": field,
            "crystallite_size_nm": size,
            TARGET_COLUMN: target,
        })

    valid = pd.DataFrame(valid_rows)
    if valid.empty:
        raise ValueError("После фильтрации не осталось валидных записей.")

    elements = sorted({
        element
        for composition in valid["composition"]
        for element in composition
        if element != "O"
    })
    stoichiometry_features = [f"comp_{element}" for element in elements]
    for element, column in zip(elements, stoichiometry_features):
        valid[column] = valid["composition"].map(
            lambda current, current_element=element: current.get(current_element, 0.0)
        )

    keys = ["composition_key", "magnetic_field_T", "crystallite_size_nm"]
    metadata = valid.groupby(keys, sort=True).agg(
        original_formula=("original_formula", "first"),
        normalized_formula_O3=("normalized_formula_O3", "first"),
        target_entropy_change=(TARGET_COLUMN, "median"),
        n_measurements=(TARGET_COLUMN, "size"),
    )
    feature_table = valid.groupby(keys, sort=True)[stoichiometry_features].first()
    dataset = metadata.join(feature_table).reset_index()

    unique_compositions = (
        valid[["composition_key", "normalized_formula_O3", "composition"]]
        .drop_duplicates("composition_key")
        .reset_index(drop=True)
    )
    return dataset, unique_compositions, pd.DataFrame(rejected_rows)


# =============================================================================
# XENONPY-ПРИЗНАКИ И МЕТАДАННЫЕ МОДЕЛИ
# =============================================================================

def calculate_xenonpy_features(unique_compositions, required_features):
    """Рассчитывает только те XenonPy-столбцы, которые ожидает модель."""
    calculators = {
        "average": WeightedAverage(n_jobs=1, on_errors="nan", return_type="df"),
        "sum": WeightedSum(n_jobs=1, on_errors="nan", return_type="df"),
        "variance": WeightedVariance(n_jobs=1, on_errors="nan", return_type="df"),
        "maximum": MaxPooling(n_jobs=1, on_errors="nan", return_type="df"),
        "minimum": MinPooling(n_jobs=1, on_errors="nan", return_type="df"),
    }
    required_set = set(required_features)
    descriptor_parts = []
    compositions = unique_compositions["composition"].tolist()

    for calculator_name, calculator in calculators.items():
        batches = []
        for start in range(0, len(compositions), XENONPY_BATCH_SIZE):
            current = calculator.transform(
                compositions[start:start + XENONPY_BATCH_SIZE]
            )
            if not isinstance(current, pd.DataFrame):
                current = pd.DataFrame(current, columns=calculator.feature_labels)
            batches.append(current.reset_index(drop=True))
        descriptors = pd.concat(batches, ignore_index=True).add_prefix(
            f"xenonpy_{calculator_name}__"
        )
        keep = [column for column in descriptors.columns if column in required_set]
        descriptor_parts.append(descriptors[keep])

    all_descriptors = pd.concat(descriptor_parts, axis=1)
    missing = sorted(required_set.difference(all_descriptors.columns))
    if missing:
        raise KeyError(
            "Не удалось воспроизвести ожидаемые XenonPy-признаки: "
            f"{missing[:10]}" + (" ..." if len(missing) > 10 else "")
        )
    return pd.concat(
        [
            unique_compositions[["composition_key"]].reset_index(drop=True),
            all_descriptors.replace([np.inf, -np.inf], np.nan),
        ],
        axis=1,
    )


def load_model_and_metadata():
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(f"Модель не найдена: {BEST_MODEL_PATH}")
    model = joblib.load(BEST_MODEL_PATH)

    metadata_candidates = [
        BEST_MODEL_PATH.with_name(f"{BEST_MODEL_PATH.stem}_metadata.json"),
        BEST_MODEL_PATH.parent / "best_overall_model_metadata.json",
    ]
    metadata = {}
    for path in metadata_candidates:
        if path.exists():
            with path.open("r", encoding="utf-8") as file:
                metadata = json.load(file)
            break

    model_features = metadata.get("model_features")
    if not model_features:
        model_features = getattr(model, "feature_names_in_", None)
    if model_features is None:
        regressor = getattr(model, "regressor_", getattr(model, "regressor", model))
        model_features = getattr(regressor, "feature_names_in_", None)
    if model_features is None:
        raise ValueError(
            "Не удалось определить входные столбцы модели. Положите рядом с "
            "моделью best_overall_model_metadata.json."
        )
    model_features = [str(value) for value in model_features]

    chemical = [
        feature for feature in model_features
        if feature not in {"magnetic_field_T", "crystallite_size_nm"}
    ]
    has_stoich = any(feature.startswith("comp_") for feature in chemical)
    has_xenonpy = any(feature.startswith("xenonpy_") for feature in chemical)
    inferred_set = (
        "Combined" if has_stoich and has_xenonpy
        else "XenonPy" if has_xenonpy
        else "Stoichiometry"
    )

    regression_pipeline = getattr(
        model, "regressor_", getattr(model, "regressor", model)
    )
    if hasattr(regression_pipeline, "named_steps"):
        model_step = regression_pipeline.named_steps.get("model")
        class_name = type(model_step).__name__
    else:
        class_name = type(regression_pipeline).__name__
    model_names = {
        "Ridge": "Ridge",
        "RandomForestRegressor": "Random Forest",
        "XGBRegressor": "XGBoost",
        "CatBoostRegressor": "CatBoost",
    }

    metadata.setdefault("feature_set", inferred_set)
    metadata.setdefault("model", model_names.get(class_name, class_name))
    metadata["model_features"] = model_features
    return model, metadata


def complete_model_inputs(dataset, unique_compositions, model_features):
    expected_xenonpy = [
        feature for feature in model_features if feature.startswith("xenonpy_")
    ]
    if expected_xenonpy:
        lookup = calculate_xenonpy_features(unique_compositions, expected_xenonpy)
        dataset = dataset.merge(
            lookup, on="composition_key", how="left", validate="many_to_one"
        )

    # Элементы, отсутствующие в конкретной выборке, имеют нулевой коэффициент.
    for feature in model_features:
        if feature.startswith("comp_") and feature not in dataset.columns:
            dataset[feature] = 0.0

    missing = sorted(set(model_features).difference(dataset.columns))
    if missing:
        raise KeyError(f"Не удалось сформировать входы модели: {missing}")
    return dataset


# =============================================================================
# ГРУППОВАЯ ВАЛИДАЦИЯ И МЕТРИКИ
# =============================================================================

def make_balanced_group_splits(group_values, n_splits, seed):
    groups = np.asarray(pd.Series(group_values).astype(str))
    unique_groups, counts = np.unique(groups, return_counts=True)
    n_splits = min(int(n_splits), len(unique_groups))
    if n_splits < 2:
        raise ValueError("Для group CV нужны минимум два уникальных состава.")

    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(unique_groups)), -counts))
    fold_groups = [[] for _ in range(n_splits)]
    fold_sizes = np.zeros(n_splits, dtype=int)
    for position in order:
        fold = int(np.argmin(fold_sizes))
        fold_groups[fold].append(unique_groups[position])
        fold_sizes[fold] += counts[position]

    splits = []
    for test_groups in fold_groups:
        test_mask = np.isin(groups, test_groups)
        train_index = np.flatnonzero(~test_mask)
        test_index = np.flatnonzero(test_mask)
        if not set(groups[train_index]).isdisjoint(set(groups[test_index])):
            raise RuntimeError("Обнаружена утечка состава между train и test.")
        splits.append((train_index, test_index))
    return splits


def physical_predictions(estimator, X):
    return np.maximum(np.asarray(estimator.predict(X), dtype=float), 0.0)


def grouped_cv_predictions(model, dataset, model_features):
    """OOF-прогнозы зафиксированной финальной конфигурации модели."""
    X = dataset[model_features].copy()
    y = dataset[TARGET_COLUMN].astype(float).copy()
    splits = make_balanced_group_splits(
        dataset["composition_key"], CV_SPLITS, RANDOM_STATE
    )
    rows = []
    for fold, (train_index, test_index) in enumerate(splits, start=1):
        estimator = clone(model)
        estimator.fit(X.iloc[train_index], y.iloc[train_index])
        prediction = physical_predictions(estimator, X.iloc[test_index])
        for local_index, global_index in enumerate(test_index):
            source = dataset.iloc[global_index]
            rows.append({
                "Feature_set": None,
                "Model": None,
                "Fold": fold,
                "Row_index": int(global_index),
                "Formula": source["normalized_formula_O3"],
                "Composition_key": source["composition_key"],
                "Magnetic_field_T": float(source["magnetic_field_T"]),
                "Crystallite_size_nm": float(source["crystallite_size_nm"]),
                "Actual": float(y.iloc[global_index]),
                "Predicted": float(prediction[local_index]),
            })
    result = pd.DataFrame(rows).sort_values("Row_index").reset_index(drop=True)
    result["Residual"] = result["Predicted"] - result["Actual"]
    return result


def locate_saved_outer_oof(metadata, dataset):
    """Ищет честные nested outer-OOF прогнозы основного расчёта."""
    expected_rows = len(dataset)
    candidates = [
        BEST_MODEL_PATH.parent.parent / "tables" / "nested_cv_oof_predictions.csv",
        BEST_MODEL_PATH.parent / "nested_cv_oof_predictions.csv",
    ]
    for path in candidates:
        if not path.exists():
            continue
        predictions = pd.read_csv(path)
        required = {
            "Feature_set", "Model", "Magnetic_field_T",
            "Crystallite_size_nm", "Actual", "Predicted",
        }
        if not required.issubset(predictions.columns):
            continue
        current = predictions.loc[
            predictions["Feature_set"].astype(str).eq(str(metadata["feature_set"]))
            & predictions["Model"].astype(str).eq(str(metadata["model"]))
        ].copy()
        if len(current) != expected_rows:
            warnings.warn(
                f"OOF-таблица {path} содержит {len(current)} подходящих строк "
                f"вместо {expected_rows}; выполняется новый group-CV расчёт."
            )
            continue
        expected_signature = (
            dataset[["magnetic_field_T", "crystallite_size_nm", TARGET_COLUMN]]
            .astype(float)
            .sort_values(["magnetic_field_T", "crystallite_size_nm", TARGET_COLUMN])
            .reset_index(drop=True)
            .to_numpy()
        )
        observed_signature = (
            current[["Magnetic_field_T", "Crystallite_size_nm", "Actual"]]
            .astype(float)
            .sort_values(["Magnetic_field_T", "Crystallite_size_nm", "Actual"])
            .reset_index(drop=True)
            .to_numpy()
        )
        if not np.allclose(
            expected_signature, observed_signature, rtol=1e-9, atol=1e-10
        ):
            warnings.warn(
                f"OOF-таблица {path} относится к другой выборке; "
                "выполняется новый group-CV расчёт."
            )
            continue
        return current.reset_index(drop=True), path
    return None, None


def safe_r2(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    if len(actual) < 2 or np.isclose(np.var(actual), 0.0):
        return np.nan
    return float(r2_score(actual, predicted))


def fieldwise_metrics(predictions, evaluation_mode):
    result = predictions.copy()
    result["Field_group_T"] = result["Magnetic_field_T"].round(
        FIELD_ROUND_DECIMALS
    )
    counts = result.groupby("Field_group_T").size()
    accepted_fields = counts[counts > MIN_SAMPLES_PER_FIELD].index
    result = result.loc[result["Field_group_T"].isin(accepted_fields)].copy()
    if result.empty:
        raise ValueError(
            f"Нет значений поля с числом записей > {MIN_SAMPLES_PER_FIELD}."
        )

    rows = []
    for field, current in result.groupby("Field_group_T", sort=True):
        actual = current["Actual"].astype(float)
        predicted = current["Predicted"].astype(float)
        rows.append({
            "Magnetic_field_T": float(field),
            "N": len(current),
            "N_unique_compositions": current.get(
                "Composition_key", pd.Series(dtype=str)
            ).nunique(),
            "R2": safe_r2(actual, predicted),
            "MAE_J_kgK": float(mean_absolute_error(actual, predicted)),
            "RMSE_J_kgK": float(mean_squared_error(actual, predicted) ** 0.5),
            "Bias_J_kgK": float((predicted - actual).mean()),
            "Target_std_J_kgK": float(actual.std(ddof=1)),
            "Target_range_J_kgK": float(actual.max() - actual.min()),
            "Mean_actual_J_kgK": float(actual.mean()),
            "Mean_predicted_J_kgK": float(predicted.mean()),
            "Evaluation_mode": evaluation_mode,
        })
    metrics = pd.DataFrame(rows).sort_values("Magnetic_field_T")
    metrics["RMSE_to_target_std"] = (
        metrics["RMSE_J_kgK"] / metrics["Target_std_J_kgK"].replace(0, np.nan)
    )
    return result, metrics


def sizewise_metrics(predictions, evaluation_mode):
    """Рассчитывает OOF-метрики в квантильных интервалах размера."""
    result = predictions.copy()
    sizes = pd.to_numeric(result["Crystallite_size_nm"], errors="coerce")
    result = result.loc[sizes.notna() & sizes.gt(0)].copy()
    number_of_bins = min(
        CRYSTALLITE_SIZE_BINS,
        int(result["Crystallite_size_nm"].nunique()),
    )
    if number_of_bins < 2:
        raise ValueError(
            "Недостаточно различных размеров кристаллитов для анализа."
        )
    result["Size_interval"] = pd.qcut(
        result["Crystallite_size_nm"],
        q=number_of_bins,
        duplicates="drop",
    )
    counts = result.groupby("Size_interval", observed=True).size()
    accepted_intervals = [
        interval
        for interval, count in counts.items()
        if count > MIN_SAMPLES_PER_SIZE_BIN
    ]
    result = result.loc[
        result["Size_interval"].isin(accepted_intervals)
    ].copy()
    if result.empty:
        raise ValueError(
            "Нет интервалов размера кристаллитов с числом записей "
            f"> {MIN_SAMPLES_PER_SIZE_BIN}."
        )

    ordered_intervals = sorted(
        result["Size_interval"].unique(), key=lambda interval: interval.left
    )
    interval_labels = {
        interval: f"S{index}"
        for index, interval in enumerate(ordered_intervals, start=1)
    }
    result["Size_group"] = result["Size_interval"].map(interval_labels)

    rows = []
    for interval in ordered_intervals:
        current = result.loc[result["Size_interval"].eq(interval)]
        actual = current["Actual"].astype(float)
        predicted = current["Predicted"].astype(float)
        rows.append({
            "Size_group": interval_labels[interval],
            "Size_min_nm": float(current["Crystallite_size_nm"].min()),
            "Size_max_nm": float(current["Crystallite_size_nm"].max()),
            "Size_median_nm": float(current["Crystallite_size_nm"].median()),
            "N": len(current),
            "N_unique_compositions": current.get(
                "Composition_key", pd.Series(dtype=str)
            ).nunique(),
            "R2": safe_r2(actual, predicted),
            "MAE_J_kgK": float(mean_absolute_error(actual, predicted)),
            "RMSE_J_kgK": float(mean_squared_error(actual, predicted) ** 0.5),
            "Bias_J_kgK": float((predicted - actual).mean()),
            "Target_std_J_kgK": float(actual.std(ddof=1)),
            "Target_range_J_kgK": float(actual.max() - actual.min()),
            "Mean_actual_J_kgK": float(actual.mean()),
            "Mean_predicted_J_kgK": float(predicted.mean()),
            "Evaluation_mode": evaluation_mode,
        })
    metrics = pd.DataFrame(rows)
    metrics["RMSE_to_target_std"] = (
        metrics["RMSE_J_kgK"] / metrics["Target_std_J_kgK"].replace(0, np.nan)
    )
    return result, metrics


# =============================================================================
# ГРАФИКИ И СОХРАНЕНИЕ
# =============================================================================

def format_field(field):
    return np.format_float_positional(float(field), trim="-")


def save_figure(fig, stem, exact_8cm=True):
    for suffix in ("png", "pdf", "svg"):
        fig.savefig(
            OUTPUT_DIRECTORY / f"{stem}.{suffix}",
            dpi=600 if suffix == "png" else None,
            bbox_inches=None if exact_8cm else "tight",
            pad_inches=0,
            facecolor="white",
        )
    plt.close(fig)


def annotate_ranked_points(axis, rows, x_column, y_column):
    """Разводит цифровые подписи и направляет крайние подписи внутрь осей."""
    x_min, x_max = axis.get_xlim()
    y_min, y_max = axis.get_ylim()
    x_span = max(x_max - x_min, np.finfo(float).eps)
    y_span = max(y_max - y_min, np.finfo(float).eps)
    vertical_offsets = (11, -13, 15, -17, 7, -9, 19, -20)
    for position, (_, row) in enumerate(rows.iterrows()):
        x_value = float(row[x_column])
        y_value = float(row[y_column])
        x_fraction = (x_value - x_min) / x_span
        y_fraction = (y_value - y_min) / y_span
        if x_fraction < 0.22:
            dx = 10 + 2 * (position % 3)
        elif x_fraction > 0.78:
            dx = -(10 + 2 * (position % 3))
        else:
            dx = 9 if position % 2 == 0 else -9
        if y_fraction < 0.16:
            dy = 12 + 2 * (position % 3)
        elif y_fraction > 0.84:
            dy = -(12 + 2 * (position % 3))
        else:
            dy = vertical_offsets[position % len(vertical_offsets)]
        axis.annotate(
            str(int(row["Error_rank"])),
            (x_value, y_value),
            xytext=(dx, dy),
            textcoords="offset points",
            ha="left" if dx >= 0 else "right",
            va="bottom" if dy >= 0 else "top",
            fontsize=6.2,
            fontweight="semibold",
            bbox={
                "boxstyle": "round,pad=0.12",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.78,
            },
            arrowprops={
                "arrowstyle": "-",
                "color": COLORS["reference"],
                "linewidth": 0.35,
                "alpha": 0.65,
            },
            annotation_clip=False,
            zorder=5,
        )


def plot_each_field(predictions, metrics, evaluation_label):
    for panel_index, metric in metrics.reset_index(drop=True).iterrows():
        field = float(metric["Magnetic_field_T"])
        current = predictions.loc[
            predictions["Field_group_T"].eq(field)
        ].copy()
        current["Absolute_error"] = (
            current["Predicted"] - current["Actual"]
        ).abs()
        actual = current["Actual"].astype(float)
        predicted = current["Predicted"].astype(float)
        lower = min(0.0, float(actual.min()), float(predicted.min()))
        upper = max(float(actual.max()), float(predicted.max()))
        padding = max(0.05 * (upper - lower), 0.1)

        fig, axis = plt.subplots(
            figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
        )
        points = axis.scatter(
            actual,
            predicted,
            c=current["Crystallite_size_nm"],
            cmap="viridis",
            s=25,
            alpha=0.86,
            edgecolor="white",
            linewidth=0.35,
        )
        axis.plot(
            [lower, upper + padding],
            [lower, upper + padding],
            linestyle="--",
            color=COLORS["reference"],
            linewidth=0.9,
        )
        is_diagnostic_field = np.isclose(
            field, DIAGNOSTIC_FIELD_T, atol=10 ** (-FIELD_ROUND_DECIMALS)
        )
        if is_diagnostic_field:
            largest_errors = current.nlargest(
                min(TOP_ERROR_LABELS, len(current)), "Absolute_error"
            ).copy()
            largest_errors["Error_rank"] = np.arange(
                1, len(largest_errors) + 1
            )
            annotate_ranked_points(
                axis, largest_errors, "Actual", "Predicted"
            )
        axis.set(
            xlim=(lower, upper + padding),
            ylim=(lower, upper + padding),
            xlabel="Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
            ylabel="Predicted |ΔSₘ| (J kg⁻¹ K⁻¹)",
        )
        axis.set_aspect("equal", adjustable="box")
        r2_text = "undefined" if pd.isna(metric["R2"]) else f"{metric['R2']:.3f}"
        axis.set_title(
            f"({chr(97 + panel_index)}) μ₀ΔH={format_field(field)} T, "
            f"n={int(metric['N'])}\n"
            f"OOF R²={r2_text}; MAE={metric['MAE_J_kgK']:.2f}",
            fontsize=8.0,
            pad=7,
        )
        axis.grid(alpha=0.20)
        colorbar = fig.colorbar(points, ax=axis, fraction=0.047, pad=0.035)
        colorbar.set_label("Crystallite size (nm)", fontsize=7)
        colorbar.ax.tick_params(labelsize=6.5)
        fig.subplots_adjust(left=0.20, right=0.88, bottom=0.16, top=0.82)

        safe_field = format_field(field).replace(".", "p").replace("-", "m")
        save_figure(
            fig,
            f"field_{safe_field}T_model_quality_8x8cm",
            exact_8cm=True,
        )


def plot_r2_vs_field(metrics, evaluation_label):
    valid = metrics.dropna(subset=["R2"]).copy()
    if valid.empty:
        warnings.warn("R² не определён ни для одного значения поля.")
        return

    fig, axis = plt.subplots(
        figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
    )
    axis.plot(
        valid["Magnetic_field_T"],
        valid["R2"],
        color=COLORS["line"],
        marker="o",
        markersize=5,
        linewidth=1.2,
    )
    axis.axhline(0, color=COLORS["reference"], linewidth=0.7)
    for point_index, (_, row) in enumerate(valid.iterrows()):
        first_point = point_index == 0
        last_point = point_index == len(valid) - 1
        horizontal_offset = 7 if first_point else -7 if last_point else 0
        horizontal_alignment = (
            "left" if first_point else "right" if last_point else "center"
        )
        axis.annotate(
            f"{row['R2']:.2f}",
            (row["Magnetic_field_T"], row["R2"]),
            xytext=(horizontal_offset, 9 if point_index % 2 == 0 else -11),
            textcoords="offset points",
            ha=horizontal_alignment,
            va="bottom" if point_index % 2 == 0 else "top",
            fontsize=6.5,
            bbox={
                "boxstyle": "round,pad=0.10",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.75,
            },
        )
    x_values = valid["Magnetic_field_T"].to_numpy(dtype=float)
    if len(x_values) == 1:
        axis.set_xlim(x_values[0] - 0.5, x_values[0] + 0.5)
    else:
        margin = max(0.04 * np.ptp(x_values), 0.12)
        axis.set_xlim(x_values.min() - margin, x_values.max() + margin)
    r2_values = valid["R2"].to_numpy(dtype=float)
    span = max(float(np.ptp(r2_values)), 0.25)
    axis.set_ylim(
        min(0.0, float(r2_values.min())) - 0.15 * span,
        max(0.10, float(r2_values.max()) + 0.28 * span),
    )
    axis.set(
        xlabel="Magnetic-field change, μ₀ΔH (T)",
        ylabel="R²",
        title="Field dependence of OOF R²",
    )
    axis.title.set_fontsize(8.5)
    axis.grid(alpha=0.22)
    fig.subplots_adjust(left=0.20, right=0.97, bottom=0.17, top=0.84)
    save_figure(fig, "r2_vs_magnetic_field_8x8cm", exact_8cm=True)


def plot_fieldwise_distribution(
    predictions,
    value_column,
    ylabel,
    title,
    stem,
):
    """Boxplot с исходными точками и выделением диагностического поля."""
    fields = sorted(predictions["Field_group_T"].unique())
    values = [
        predictions.loc[
            predictions["Field_group_T"].eq(field), value_column
        ].astype(float).to_numpy()
        for field in fields
    ]
    labels = [
        f"{format_field(field)}\n(n={len(current_values)})"
        for field, current_values in zip(fields, values)
    ]

    fig, axis = plt.subplots(
        figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
    )
    boxplot = axis.boxplot(
        values,
        labels=labels,
        patch_artist=True,
        widths=0.58,
        showfliers=False,
        medianprops={"color": COLORS["reference"], "linewidth": 1.1},
        whiskerprops={"color": COLORS["reference"], "linewidth": 0.8},
        capprops={"color": COLORS["reference"], "linewidth": 0.8},
    )
    for patch, field in zip(boxplot["boxes"], fields):
        patch.set_facecolor(
            COLORS["line"]
            if np.isclose(field, DIAGNOSTIC_FIELD_T)
            else COLORS["points"]
        )
        patch.set_alpha(0.78)
        patch.set_edgecolor("white")

    rng = np.random.default_rng(RANDOM_STATE)
    for position, current_values in enumerate(values, start=1):
        jitter = rng.uniform(-0.14, 0.14, size=len(current_values))
        axis.scatter(
            np.full(len(current_values), position) + jitter,
            current_values,
            s=8,
            color=COLORS["reference"],
            alpha=0.24,
            linewidth=0,
            zorder=1,
        )

    axis.set(
        xlabel="Magnetic-field change, μ₀ΔH (T)",
        ylabel=ylabel,
        title=title,
    )
    axis.title.set_fontsize(8.5)
    axis.tick_params(axis="x", labelsize=7)
    axis.grid(axis="y", alpha=0.22)
    axis.set_axisbelow(True)
    fig.subplots_adjust(left=0.21, right=0.97, bottom=0.18, top=0.88)
    save_figure(fig, stem, exact_8cm=True)


def diagnostic_error_table(predictions):
    tolerance = 10 ** (-FIELD_ROUND_DECIMALS)
    current = predictions.loc[
        np.isclose(
            predictions["Field_group_T"],
            DIAGNOSTIC_FIELD_T,
            atol=tolerance,
        )
    ].copy()
    if current.empty:
        warnings.warn(
            f"Поле {DIAGNOSTIC_FIELD_T:g} Т не прошло условие "
            f"N > {MIN_SAMPLES_PER_FIELD}; диагностика пропущена."
        )
        return pd.DataFrame()

    current["Residual_J_kgK"] = current["Predicted"] - current["Actual"]
    current["Absolute_error_J_kgK"] = current["Residual_J_kgK"].abs()
    current = current.sort_values(
        "Absolute_error_J_kgK", ascending=False
    ).reset_index(drop=True)
    current.insert(0, "Error_rank", np.arange(1, len(current) + 1))
    preferred_columns = [
        "Error_rank",
        "Formula",
        "Composition_key",
        "Fold",
        "Magnetic_field_T",
        "Crystallite_size_nm",
        "Actual",
        "Predicted",
        "Residual_J_kgK",
        "Absolute_error_J_kgK",
    ]
    return current[
        [column for column in preferred_columns if column in current.columns]
    ]


def plot_diagnostic_residuals(predictions, error_table, evaluation_label):
    """Два независимых residual-графика для выбранного поля."""
    if error_table.empty:
        return
    tolerance = 10 ** (-FIELD_ROUND_DECIMALS)
    current = predictions.loc[
        np.isclose(
            predictions["Field_group_T"],
            DIAGNOSTIC_FIELD_T,
            atol=tolerance,
        )
    ].copy()
    current["Residual_J_kgK"] = current["Predicted"] - current["Actual"]

    label_rows = error_table.head(
        min(TOP_ERROR_LABELS, len(error_table))
    ).copy()

    specifications = [
        {
            "x": "Actual",
            "color": "Crystallite_size_nm",
            "xlabel": "Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
            "colorbar": "Crystallite size (nm)",
            "stem": (
                f"diagnostic_{format_field(DIAGNOSTIC_FIELD_T).replace('.', 'p')}T_"
                "residual_vs_measured_8x8cm"
            ),
        },
        {
            "x": "Crystallite_size_nm",
            "color": "Actual",
            "xlabel": "Crystallite size (nm)",
            "colorbar": "Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
            "stem": (
                f"diagnostic_{format_field(DIAGNOSTIC_FIELD_T).replace('.', 'p')}T_"
                "residual_vs_crystallite_size_8x8cm"
            ),
        },
    ]

    for panel_index, specification in enumerate(specifications):
        fig, axis = plt.subplots(
            figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
        )
        points = axis.scatter(
            current[specification["x"]],
            current["Residual_J_kgK"],
            c=current[specification["color"]],
            cmap="viridis",
            s=26,
            alpha=0.86,
            edgecolor="white",
            linewidth=0.35,
        )
        axis.axhline(
            0,
            color=COLORS["reference"],
            linestyle="--",
            linewidth=0.9,
        )
        annotate_ranked_points(
            axis,
            label_rows,
            specification["x"],
            "Residual_J_kgK",
        )
        axis.set(
            xlabel=specification["xlabel"],
            ylabel="Residual, predicted - measured (J kg⁻¹ K⁻¹)",
            title=(
                f"({chr(97 + panel_index)}) Residuals at "
                f"{format_field(DIAGNOSTIC_FIELD_T)} T\n"
                "Numbers mark the largest |errors|"
            ),
        )
        axis.margins(x=0.06, y=0.14)
        axis.title.set_fontsize(8.0)
        axis.grid(alpha=0.20)
        colorbar = fig.colorbar(points, ax=axis, fraction=0.047, pad=0.035)
        colorbar.set_label(specification["colorbar"], fontsize=7)
        colorbar.ax.tick_params(labelsize=6.5)
        fig.subplots_adjust(left=0.22, right=0.88, bottom=0.17, top=0.82)
        save_figure(fig, specification["stem"], exact_8cm=True)


def create_diagnostic_plots(predictions, evaluation_label):
    diagnostic_data = predictions.copy()
    diagnostic_data["Absolute_error_J_kgK"] = (
        diagnostic_data["Predicted"] - diagnostic_data["Actual"]
    ).abs()
    plot_fieldwise_distribution(
        diagnostic_data,
        value_column="Actual",
        ylabel="Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
        title="Target distribution (2 T highlighted)",
        stem="target_distribution_by_field_8x8cm",
    )
    plot_fieldwise_distribution(
        diagnostic_data,
        value_column="Absolute_error_J_kgK",
        ylabel="Absolute OOF error (J kg⁻¹ K⁻¹)",
        title="Absolute OOF error (2 T highlighted)",
        stem="absolute_error_by_field_8x8cm",
    )
    error_table = diagnostic_error_table(diagnostic_data)
    plot_diagnostic_residuals(
        diagnostic_data, error_table, evaluation_label
    )
    return error_table


def worst_size_group(size_metrics):
    valid = size_metrics.dropna(subset=["R2"])
    if not valid.empty:
        return str(valid.sort_values("R2").iloc[0]["Size_group"])
    return str(size_metrics.sort_values("MAE_J_kgK", ascending=False).iloc[0]["Size_group"])


def size_group_tick_labels(size_metrics):
    labels = {}
    for _, row in size_metrics.iterrows():
        labels[str(row["Size_group"])] = (
            f"{row['Size_group']}\n"
            f"{row['Size_min_nm']:.0f}-{row['Size_max_nm']:.0f}"
        )
    return labels


def plot_r2_vs_crystallite_size(size_metrics):
    valid = size_metrics.dropna(subset=["R2"]).copy()
    if valid.empty:
        warnings.warn("R² не определён ни для одного интервала размера.")
        return
    fig, axis = plt.subplots(
        figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
    )
    axis.plot(
        valid["Size_median_nm"],
        valid["R2"],
        color=COLORS["line"],
        marker="o",
        markersize=5,
        linewidth=1.2,
    )
    axis.axhline(0, color=COLORS["reference"], linewidth=0.7)
    for position, (_, row) in enumerate(valid.iterrows()):
        first_point = position == 0
        last_point = position == len(valid) - 1
        horizontal_offset = 8 if first_point else -8 if last_point else 0
        horizontal_alignment = (
            "left" if first_point else "right" if last_point else "center"
        )
        axis.annotate(
            f"{row['R2']:.2f}",
            (row["Size_median_nm"], row["R2"]),
            xytext=(horizontal_offset, 9 if position % 2 == 0 else -11),
            textcoords="offset points",
            ha=horizontal_alignment,
            va="bottom" if position % 2 == 0 else "top",
            fontsize=6.3,
            bbox={
                "boxstyle": "round,pad=0.10",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.78,
            },
        )
    r2_values = valid["R2"].to_numpy(dtype=float)
    r2_span = max(float(np.ptp(r2_values)), 0.25)
    axis.set_ylim(
        min(0.0, float(r2_values.min())) - 0.16 * r2_span,
        max(0.10, float(r2_values.max()) + 0.30 * r2_span),
    )
    axis.margins(x=0.08)
    axis.set(
        xlabel="Median crystallite size (nm)",
        ylabel="R²",
        title="OOF R² by crystallite size",
    )
    axis.title.set_fontsize(8.3)
    axis.grid(alpha=0.22)
    fig.subplots_adjust(left=0.20, right=0.97, bottom=0.17, top=0.85)
    save_figure(fig, "r2_vs_crystallite_size_8x8cm", exact_8cm=True)


def plot_sizewise_distribution(
    size_predictions,
    size_metrics,
    value_column,
    ylabel,
    title,
    stem,
):
    groups = size_metrics["Size_group"].astype(str).tolist()
    worst_group = worst_size_group(size_metrics)
    label_lookup = size_group_tick_labels(size_metrics)
    values = [
        size_predictions.loc[
            size_predictions["Size_group"].astype(str).eq(group), value_column
        ].astype(float).to_numpy()
        for group in groups
    ]
    labels = [label_lookup[group] for group in groups]

    fig, axis = plt.subplots(
        figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
    )
    boxplot = axis.boxplot(
        values,
        labels=labels,
        patch_artist=True,
        widths=0.58,
        showfliers=False,
        medianprops={"color": COLORS["reference"], "linewidth": 1.1},
        whiskerprops={"color": COLORS["reference"], "linewidth": 0.8},
        capprops={"color": COLORS["reference"], "linewidth": 0.8},
    )
    for patch, group in zip(boxplot["boxes"], groups):
        patch.set_facecolor(
            COLORS["line"] if group == worst_group else COLORS["points"]
        )
        patch.set_alpha(0.78)
        patch.set_edgecolor("white")

    rng = np.random.default_rng(RANDOM_STATE + 1)
    for position, current_values in enumerate(values, start=1):
        jitter = rng.uniform(-0.14, 0.14, size=len(current_values))
        axis.scatter(
            np.full(len(current_values), position) + jitter,
            current_values,
            s=8,
            color=COLORS["reference"],
            alpha=0.23,
            linewidth=0,
            zorder=1,
        )
    axis.set(
        xlabel="Crystallite-size interval (nm)",
        ylabel=ylabel,
        title=title,
    )
    axis.title.set_fontsize(8.3)
    axis.tick_params(axis="x", labelsize=6.2)
    axis.grid(axis="y", alpha=0.22)
    axis.set_axisbelow(True)
    fig.subplots_adjust(left=0.21, right=0.97, bottom=0.20, top=0.86)
    save_figure(fig, stem, exact_8cm=True)


def size_error_table(size_predictions, size_metrics):
    diagnostic_group = worst_size_group(size_metrics)
    current = size_predictions.loc[
        size_predictions["Size_group"].astype(str).eq(diagnostic_group)
    ].copy()
    current["Residual_J_kgK"] = current["Predicted"] - current["Actual"]
    current["Absolute_error_J_kgK"] = current["Residual_J_kgK"].abs()
    current = current.sort_values(
        "Absolute_error_J_kgK", ascending=False
    ).reset_index(drop=True)
    current.insert(0, "Error_rank", np.arange(1, len(current) + 1))
    metric = size_metrics.loc[
        size_metrics["Size_group"].astype(str).eq(diagnostic_group)
    ].iloc[0]
    current.insert(1, "Diagnostic_size_group", diagnostic_group)
    current.insert(2, "Group_size_min_nm", float(metric["Size_min_nm"]))
    current.insert(3, "Group_size_max_nm", float(metric["Size_max_nm"]))
    return current


def plot_worst_size_group_residuals(
    size_predictions,
    size_metrics,
    error_table,
):
    if error_table.empty:
        return
    diagnostic_group = str(error_table.iloc[0]["Diagnostic_size_group"])
    metric = size_metrics.loc[
        size_metrics["Size_group"].astype(str).eq(diagnostic_group)
    ].iloc[0]
    current = size_predictions.loc[
        size_predictions["Size_group"].astype(str).eq(diagnostic_group)
    ].copy()
    current["Residual_J_kgK"] = current["Predicted"] - current["Actual"]
    labels = error_table.head(min(TOP_ERROR_LABELS, len(error_table))).copy()

    fig, axis = plt.subplots(
        figsize=(FIGURE_SIZE_INCH, FIGURE_SIZE_INCH)
    )
    points = axis.scatter(
        current["Actual"],
        current["Residual_J_kgK"],
        c=current["Magnetic_field_T"],
        cmap="viridis",
        s=26,
        alpha=0.86,
        edgecolor="white",
        linewidth=0.35,
    )
    axis.axhline(
        0,
        color=COLORS["reference"],
        linestyle="--",
        linewidth=0.9,
    )
    annotate_ranked_points(
        axis, labels, "Actual", "Residual_J_kgK"
    )
    axis.set(
        xlabel="Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
        ylabel="Residual, predicted - measured (J kg⁻¹ K⁻¹)",
        title=(
            f"Residuals for {diagnostic_group}: "
            f"{metric['Size_min_nm']:.0f}-{metric['Size_max_nm']:.0f} nm\n"
            "Numbers mark the largest |errors|"
        ),
    )
    axis.margins(x=0.07, y=0.15)
    axis.title.set_fontsize(8.0)
    axis.grid(alpha=0.20)
    colorbar = fig.colorbar(points, ax=axis, fraction=0.047, pad=0.035)
    colorbar.set_label("Magnetic-field change, μ₀ΔH (T)", fontsize=7)
    colorbar.ax.tick_params(labelsize=6.5)
    fig.subplots_adjust(left=0.22, right=0.88, bottom=0.17, top=0.82)
    save_figure(
        fig,
        f"diagnostic_{diagnostic_group.lower()}_residual_vs_measured_8x8cm",
        exact_8cm=True,
    )


def create_size_diagnostic_plots(predictions, evaluation_mode):
    size_predictions, size_metrics = sizewise_metrics(
        predictions, evaluation_mode
    )
    size_predictions["Absolute_error_J_kgK"] = (
        size_predictions["Predicted"] - size_predictions["Actual"]
    ).abs()
    plot_r2_vs_crystallite_size(size_metrics)
    plot_sizewise_distribution(
        size_predictions,
        size_metrics,
        value_column="Actual",
        ylabel="Measured |ΔSₘ| (J kg⁻¹ K⁻¹)",
        title="Target distribution by crystallite size",
        stem="target_distribution_by_crystallite_size_8x8cm",
    )
    plot_sizewise_distribution(
        size_predictions,
        size_metrics,
        value_column="Absolute_error_J_kgK",
        ylabel="Absolute OOF error (J kg⁻¹ K⁻¹)",
        title="Absolute OOF error by crystallite size",
        stem="absolute_error_by_crystallite_size_8x8cm",
    )
    error_table = size_error_table(size_predictions, size_metrics)
    plot_worst_size_group_residuals(
        size_predictions, size_metrics, error_table
    )
    return size_predictions, size_metrics, error_table


def summarize_size_dependence(size_metrics):
    """Компактная таблица для формулировки выводов по размеру."""
    valid = size_metrics.dropna(subset=["R2"]).copy()
    if valid.empty:
        return pd.DataFrame()
    worst_r2 = valid.sort_values("R2").iloc[0]
    best_r2 = valid.sort_values("R2", ascending=False).iloc[0]
    largest_mae = valid.sort_values("MAE_J_kgK", ascending=False).iloc[0]
    size_r2_correlation = (
        float(valid["Size_median_nm"].corr(valid["R2"]))
        if len(valid) >= 3 else np.nan
    )
    size_mae_correlation = (
        float(valid["Size_median_nm"].corr(valid["MAE_J_kgK"]))
        if len(valid) >= 3 else np.nan
    )
    return pd.DataFrame([{
        "Worst_R2_group": worst_r2["Size_group"],
        "Worst_R2_size_min_nm": worst_r2["Size_min_nm"],
        "Worst_R2_size_max_nm": worst_r2["Size_max_nm"],
        "Worst_R2_N": int(worst_r2["N"]),
        "Worst_R2": worst_r2["R2"],
        "Worst_group_MAE_J_kgK": worst_r2["MAE_J_kgK"],
        "Worst_group_RMSE_J_kgK": worst_r2["RMSE_J_kgK"],
        "Worst_group_bias_J_kgK": worst_r2["Bias_J_kgK"],
        "Worst_group_target_std_J_kgK": worst_r2["Target_std_J_kgK"],
        "Worst_group_RMSE_to_target_std": worst_r2["RMSE_to_target_std"],
        "Best_R2_group": best_r2["Size_group"],
        "Best_R2": best_r2["R2"],
        "Largest_MAE_group": largest_mae["Size_group"],
        "Largest_MAE_J_kgK": largest_mae["MAE_J_kgK"],
        "Pearson_median_size_vs_R2": size_r2_correlation,
        "Pearson_median_size_vs_MAE": size_mae_correlation,
    }])


def build_size_conclusion_text(size_metrics, size_summary):
    """Формирует краткий data-driven вывод, пригодный как черновик для статьи."""
    if size_metrics.empty or size_summary.empty:
        return "Crystallite-size-stratified performance could not be evaluated."
    summary = size_summary.iloc[0]
    worst = size_metrics.loc[
        size_metrics["Size_group"].astype(str).eq(
            str(summary["Worst_R2_group"])
        )
    ].iloc[0]
    best = size_metrics.loc[
        size_metrics["Size_group"].astype(str).eq(
            str(summary["Best_R2_group"])
        )
    ].iloc[0]

    def correlation_phrase(value, metric):
        if pd.isna(value):
            return f"the correlation with {metric} was not estimable"
        direction = "increased" if value > 0 else "decreased"
        return (
            f"{metric} {direction} with median crystallite size "
            f"(Pearson r = {value:.2f})"
        )

    return (
        "Crystallite-size-stratified OOF performance was lowest for "
        f"{worst['Size_group']} ({worst['Size_min_nm']:.0f}-"
        f"{worst['Size_max_nm']:.0f} nm; n = {int(worst['N'])}), with "
        f"R² = {worst['R2']:.3f}, MAE = {worst['MAE_J_kgK']:.3f}, "
        f"RMSE = {worst['RMSE_J_kgK']:.3f}, and bias = "
        f"{worst['Bias_J_kgK']:.3f} J kg⁻¹ K⁻¹. The highest R² was "
        f"obtained for {best['Size_group']} ({best['Size_min_nm']:.0f}-"
        f"{best['Size_max_nm']:.0f} nm; R² = {best['R2']:.3f}). Across "
        "the quantile intervals, "
        f"{correlation_phrase(summary['Pearson_median_size_vs_R2'], 'R²')}, "
        "whereas "
        f"{correlation_phrase(summary['Pearson_median_size_vs_MAE'], 'MAE')}. "
        "These subgroup associations are descriptive and should not be "
        "interpreted as causal effects of crystallite size."
    )


def save_tables(
    predictions,
    metrics,
    size_predictions,
    size_metrics,
    size_summary,
    rejected,
    metadata,
    evaluation_mode,
    diagnostic_errors,
    size_diagnostic_errors,
):
    predictions.to_csv(
        OUTPUT_DIRECTORY / "fieldwise_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    metrics.to_csv(
        OUTPUT_DIRECTORY / "fieldwise_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )
    size_predictions.to_csv(
        OUTPUT_DIRECTORY / "sizewise_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    size_metrics.to_csv(
        OUTPUT_DIRECTORY / "sizewise_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )
    size_summary.to_csv(
        OUTPUT_DIRECTORY / "crystallite_size_diagnostic_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    rejected.to_csv(
        OUTPUT_DIRECTORY / "rejected_rows.csv",
        index=False,
        encoding="utf-8-sig",
    )
    diagnostic_errors.to_csv(
        OUTPUT_DIRECTORY
        / f"largest_errors_at_{format_field(DIAGNOSTIC_FIELD_T)}T.csv",
        index=False,
        encoding="utf-8-sig",
    )
    size_diagnostic_errors.to_csv(
        OUTPUT_DIRECTORY / "largest_errors_in_worst_size_group.csv",
        index=False,
        encoding="utf-8-sig",
    )
    run_info = pd.DataFrame([{
        "Input_file": str(INPUT_FILE),
        "Best_model_path": str(BEST_MODEL_PATH),
        "Feature_set": metadata.get("feature_set"),
        "Model": metadata.get("model"),
        "Evaluation_mode": evaluation_mode,
        "Diagnostic_field_T": DIAGNOSTIC_FIELD_T,
        "Evaluation_note": (
            "Exact predictions from nested outer folds"
            if evaluation_mode == "saved_nested_outer_OOF"
            else "Grouped CV of a configuration previously selected on the full study"
        ),
        "Minimum_required_condition": f"N > {MIN_SAMPLES_PER_FIELD}",
    }])
    with pd.ExcelWriter(
        OUTPUT_DIRECTORY / "fieldwise_model_quality.xlsx", engine="openpyxl"
    ) as writer:
        metrics.to_excel(writer, sheet_name="Metrics_by_field", index=False)
        predictions.to_excel(writer, sheet_name="Predictions", index=False)
        size_metrics.to_excel(writer, sheet_name="Metrics_by_size", index=False)
        size_summary.to_excel(writer, sheet_name="Size_summary", index=False)
        size_predictions.to_excel(
            writer, sheet_name="Sizewise_predictions", index=False
        )
        rejected.to_excel(writer, sheet_name="Rejected_rows", index=False)
        diagnostic_errors.to_excel(
            writer, sheet_name="Diagnostic_field_errors", index=False
        )
        size_diagnostic_errors.to_excel(
            writer, sheet_name="Diagnostic_size_errors", index=False
        )
        run_info.to_excel(writer, sheet_name="Run_info", index=False)


# =============================================================================
# ЗАПУСК
# =============================================================================

def main():
    global OUTPUT_DIRECTORY
    if OUTPUT_DIRECTORY is None:
        OUTPUT_DIRECTORY = BEST_MODEL_PATH.parent.parent / "fieldwise_quality"
    else:
        OUTPUT_DIRECTORY = Path(OUTPUT_DIRECTORY)
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    print("Загрузка модели:", BEST_MODEL_PATH)
    model, metadata = load_model_and_metadata()
    model_features = metadata["model_features"]
    print("Представление признаков:", metadata["feature_set"])
    print("Модель:", metadata["model"])

    source = load_raw_source()
    dataset, unique_compositions, rejected = prepare_ml_dataset(source)
    dataset = complete_model_inputs(dataset, unique_compositions, model_features)
    print("Валидных агрегированных записей:", len(dataset))

    predictions, oof_path = locate_saved_outer_oof(metadata, dataset)
    if predictions is not None:
        evaluation_mode = "saved_nested_outer_OOF"
        evaluation_label = "Nested outer OOF"
        print("Используются сохранённые outer-OOF прогнозы:", oof_path)
    else:
        evaluation_mode = "group_CV_of_frozen_final_configuration"
        evaluation_label = "Grouped OOF"
        print(
            "Сохранённые nested OOF-прогнозы не найдены; "
            "выполняется group CV зафиксированной конфигурации."
        )
        predictions = grouped_cv_predictions(model, dataset, model_features)
        predictions["Feature_set"] = metadata["feature_set"]
        predictions["Model"] = metadata["model"]

    accepted_predictions, metrics = fieldwise_metrics(
        predictions, evaluation_mode
    )
    plot_each_field(accepted_predictions, metrics, evaluation_label)
    plot_r2_vs_field(metrics, evaluation_label)
    diagnostic_errors = create_diagnostic_plots(
        accepted_predictions, evaluation_label
    )
    (
        size_predictions,
        size_metrics,
        size_diagnostic_errors,
    ) = create_size_diagnostic_plots(
        predictions, evaluation_mode
    )
    size_summary = summarize_size_dependence(size_metrics)
    size_conclusion = build_size_conclusion_text(
        size_metrics, size_summary
    )
    (OUTPUT_DIRECTORY / "crystallite_size_conclusions.txt").write_text(
        size_conclusion + "\n", encoding="utf-8"
    )
    save_tables(
        accepted_predictions,
        metrics,
        size_predictions,
        size_metrics,
        size_summary,
        rejected,
        metadata,
        evaluation_mode,
        diagnostic_errors,
        size_diagnostic_errors,
    )

    print("\nКачество по значениям поля (используются только N > 10):")
    print(metrics.to_string(index=False))
    print("\nКачество по интервалам размера кристаллитов:")
    print(size_metrics.to_string(index=False))
    print("\nСводка для выводов по размеру кристаллитов:")
    print(size_summary.to_string(index=False))
    print("\nАвтоматический вывод по размеру кристаллитов:")
    print(size_conclusion)
    print("\nРезультаты сохранены:", OUTPUT_DIRECTORY.resolve())
    return {
        "metrics": metrics,
        "predictions": accepted_predictions,
        "metadata": metadata,
        "evaluation_mode": evaluation_mode,
        "diagnostic_errors": diagnostic_errors,
        "size_metrics": size_metrics,
        "size_predictions": size_predictions,
        "size_diagnostic_errors": size_diagnostic_errors,
        "size_summary": size_summary,
        "size_conclusion": size_conclusion,
    }


if __name__ == "__main__":
    results = main()


# In[ ]:


Загрузка модели: C:\Users\d.kanurin\Desktop\РНФ.ИРАН\Научка\DCD_paper\Data\MCE_Three_Feature_Sets_GridSearch_results\models\best_overall_model.joblib
Представление признаков: Stoichiometry
Модель: CatBoost
Валидных агрегированных записей: 259
Используются сохранённые outer-OOF прогнозы: C:\Users\d.kanurin\Desktop\РНФ.ИРАН\Научка\DCD_paper\Data\MCE_Three_Feature_Sets_GridSearch_results\tables\nested_cv_oof_predictions.csv

Качество по значениям поля (используются только N > 10):
 Magnetic_field_T   N  N_unique_compositions        R2  MAE_J_kgK  RMSE_J_kgK  Bias_J_kgK  Target_std_J_kgK  Target_range_J_kgK  Mean_actual_J_kgK  Mean_predicted_J_kgK        Evaluation_mode  RMSE_to_target_std
              1.0  42                     36  0.489626   0.271609    0.383178    0.004343          0.542861                2.18           1.104095              1.108438 saved_nested_outer_OOF            0.705848
              2.0  35                     35 -0.188086   0.822626    1.045005   -0.110012    